# Real-Time Bitcoin Transaction Anomaly Detection using Anthropic Claude

## Introduction

This project builds a real-time Bitcoin transaction anomaly detection system using Anthropic Claude. It fetches live transaction data from the Blockchair API, extracts structural features in real time, and prepares transactions for anomaly analysis and LLM-based explanations.

The system is modular: real-time ingestion → feature engineering → anomaly detection → natural language explanation → alerting → dashboard visualization.

## Objective

- Ingest real-time Bitcoin transaction data from the Blockchair API.
- Extract structural and behavioral features from transactions.
- Apply a two-stage anomaly detection system (rule-based and LLM-based).
- Generate interpretable explanations for flagged transactions using Anthropic Claude.
- Visualize the results through dashboards and integrate alerts.
- Prepare the system for scalable deployment using Spark and AWS.

## Project Workflow

1. Data Ingestion from Blockchair API
2. Feature Engineering and Time Bucketing
3. Anomaly Detection:
   - Rule-based Filtering
   - Claude LLM Analysis
4. Temporal Pattern Detection
5. Alerting and Dashboarding
6. Deployment and Auto-scaling
7. Testing and Final Reporting

## 1. Integrated Real-Time Pipeline Overview

The full pipeline is split across the following modules:

- **extract_and_upload.py**: 
  - Pulls Bitcoin transactions from Blockchair API.
  - Uploads small structured batches to AWS S3 under `/raw/` folder.

- **preprocess_with_spark.py**:
  - Loads raw transactions from S3.
  - Applies feature engineering, windowed aggregation (1min/5min), and synthetic oversampling using PySpark.
  - Saves processed and balanced data to S3 under `/processed/` folder.

- **anomaly_explainer.py**:
  - Pulls processed anomalies from S3.
  - Generates chain-of-thought style human-readable explanations using Anthropic Claude.
  - Uploads final output JSON files to `/explained/` folder in S3.

## 2. High-Level Architecture

Blockchair API → (extract_and_upload.py) → S3 (/raw/)

         ↓
(preprocess_with_spark.py)

         ↓
S3 (/processed/)

         ↓
(anomaly_explainer.py)

         ↓
S3 (/explained/)

## Important Notes Before Running the Pipeline

- Ensure AWS credentials (`aws configure`) are properly set on your machine.
- Ensure the AWS S3 bucket `btc-anomaly` is created and accessible.
- Ensure Anthropic Claude API key is available and exported as environment variable `ANTHROPIC_API_KEY`.
- Install all required Python packages:
    - `boto3`
    - `s3fs`
    - `pyarrow`
    - `anthropic`
    - `pyspark`
- Install Hadoop-AWS support for Spark (`hadoop-aws` JAR) when running Spark jobs.

Running all code cells sequentially will automatically extract data, preprocess it, and generate explanations.

In [7]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Load .env from parent directory
env_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path=env_path)

AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY")
CLAUDE_KEY = os.getenv("ANTHROPIC_API_KEY")

In [8]:
import os

# Actual AWS credentials
os.environ['AWS_ACCESS_KEY_ID'] = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET

In [9]:
import anthropic

from anthropic import Anthropic

client = Anthropic(api_key=CLAUDE_KEY)

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BTC Preprocessing with Aggregation") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ['AWS_ACCESS_KEY_ID']) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ['AWS_SECRET_ACCESS_KEY']) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .getOrCreate()

25/05/08 14:24:53 WARN Utils: Your hostname, Anveshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.124 instead (on interface en0)
25/05/08 14:24:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/08 14:24:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
import subprocess

# Step 1: Run data extraction script
try:
    subprocess.run(["python", "../scripts/extract_and_upload.py"], check=True)
    print("Data extraction completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during data extraction:", e)

/Users/chitturi/Documents/DATA 606/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/notebooks/../scripts/extract_and_upload.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


Batch 1
Uploaded to S3: s3://btc-anomaly/raw/2025/05/08/batch_20250508_182455.json
Data extraction completed successfully.


In [12]:
# Step 2: Run preprocessing script using spark-submit
try:
    subprocess.run([
        "spark-submit",
        "--packages",
        "org.apache.hadoop:hadoop-aws:3.3.2",
        "../scripts/preprocess_with_spark.py"
    ], check=True)
    print("Data preprocessing completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during data preprocessing:", e)

25/05/08 14:25:01 WARN Utils: Your hostname, Anveshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.124 instead (on interface en0)
25/05/08 14:25:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/chitturi/.ivy2/cache
The jars for the packages stored in: /Users/chitturi/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-34b58f3b-d235-4a01-b36f-dcf1e7239b5e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 82ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [def

:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


25/05/08 14:25:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/08 14:25:02 INFO SparkContext: Running Spark version 3.5.5
25/05/08 14:25:02 INFO SparkContext: OS info Mac OS X, 15.4.1, aarch64
25/05/08 14:25:02 INFO SparkContext: Java version 11.0.26
25/05/08 14:25:02 INFO ResourceUtils: ==============================================================
25/05/08 14:25:02 INFO ResourceUtils: No custom resources configured for spark.driver.
25/05/08 14:25:02 INFO ResourceUtils: ==============================================================
25/05/08 14:25:02 INFO SparkContext: Submitted application: BTC Preprocessing
25/05/08 14:25:02 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> na

Data preprocessing completed successfully.


In [13]:
# Step 3: Run anomaly explanation script
try:
    subprocess.run(["python", "../scripts/anomaly_explainer.py"], check=True)
    print("Anomaly explanation completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during anomaly explanation:", e)


Explanation for transaction 1:
[TextBlock(citations=None, text="Here is an analysis of the provided Bitcoin transaction data:\n\nTransaction Details:\n- Timestamp window: 1746741960000000000 to 1746742020000000000 (60 second interval) \n- Transaction count in the 1 minute window: 50.0 transactions\n\nDeviation from Account History:\n- Without more historical data on the account's typical transaction volumes, it's difficult to definitively say if 50 transactions per minute is abnormal for this specific account. More context would be needed to assess deviation from the account's own baseline.\n\nComparison to Network-Wide Baselines:\n- As of 2023, the Bitcoin network processes roughly 2-4 transactions per second on average network-wide. This equates to approximately 120-240 transactions per minute across the entire Bitcoin blockchain.\n- 50 transactions attributed to a single account in one minute is quite high compared to the network-wide average of a few transactions per second. It re